In [ ]:
import pandas as pd
import json
import re

In [ ]:
#this function grabs the upload date and church name from the .json files
#the upload date is within the actual JSON
#the church name comes from the file path (you could do this differently if you want)
def get_metadata(df_file_name):
    a=df_file_name
    base_file=''
    next=False
    church_name=''
    for x in a.rsplit('\\')[:-1]:
        if x=="Church_NLP":
            next=True
        elif next==True:
            next=False
            church_name=x
        base_file+=x+'\\'
    try:
        file_end='audio_'+a.rsplit('\\')[-1].split('_')[0]+'.wav.info.json'
        with open(base_file+file_end, 'r', encoding="utf8") as file:
            data = json.load(file)
    except:
        file_end=a.rsplit('\\')[-1].split('_')[0]+'.wav.info.json'
        with open(base_file+file_end, 'r', encoding="utf8") as file:
            data = json.load(file)
    return data['upload_date'],church_name

In [ ]:
#this cell appends the church names and upload date to the results dataframe
df=pd.read_csv("~filepath\\structured_output1.csv")

dates=[]
church_name=[]
for i in df['files']:
    try:
        a,b=get_metadata(i)
        church_name.append(b)
        dates.append(a)
    except:
        dates.append('nan')
        church_name.append('nan')
df['date']=dates
df['church_name']=church_name
df=df.fillna('nan')

df.head()


In [ ]:
#the rest of this notebook contains functions that were used to draw findings out from the data
#findings themselves are not present in this notebook
#In other words, these functions create the dataframe that you could feed to google sheets and build a chart from there
#the output to functions is commented out but obviously you should uncomment the outputs you want to see

In [ ]:
#this function gets top songs combined across all sections but only from services after a certain date
#The minimum date may not be needed if all your data is the same timeframe
def get_top_songs_all_data(df):
    song_list=[]
    df1=df[df['date']>'20241231']
    for col in ['song1','song2','song3','song4','song5','song6','song7','song8','song9','song10']:
        for i in df1[col]:
            if i != 'nan':
                for x in i.split(','):
                    song_list.append(re.sub(r"^[ ]+|[^a-zA-Z ]+", "", x).lower())
    song_df=pd.DataFrame()
    song_df['songs']=song_list
    a=song_df['songs'].value_counts().reset_index()
    a=a[a['count']>0]
    return a
#a=get_top_songs_all_data(df)

#similar to the last function, but this one groups by month
#this actually groups by year and month, but fed in the example is designed to be from the same year
def get_top_songs_month(df):
    list_dfs=[]
    list_dates=[]
    df1=df
    df1['date']=df['date'].apply(lambda x: x[:6])
    for i in list(df1['date'].unique()):
        song_list=[]
        df1=df
        df1['date']=df['date'].apply(lambda x: x[:6])
        df1=df1[df1['date']==i]
        if df1.shape[0]>5:
            list_dates.append(i)
            for col in ['song1','song2','song3','song4','song5','song6','song7','song8','song9','song10']:
                for i in df1[col]:
                    if i != 'nan':
                        for x in i.split(','):
                            song_list.append(re.sub(r"^[ ]+|[^a-zA-Z ]+", "", x).lower())
            song_df=pd.DataFrame()
            song_df['songs']=song_list
            a=song_df['songs'].value_counts().reset_index()
            a=a[a['count']>3]
            list_dfs.append(a)
    return list_dfs, list_dates
#a,b=get_top_songs_month(df)

#this function counts the number of distinct songs and number of occurrences and groups by church name
def church_song_diversity(df):
    churches=list(df['church_name'].unique())
    list_of_churches=[]
    list_dfs=[]
    for i in churches:
        df1=df[df['church_name']==i]
        song_list=[]
        if df1.shape[0]>5:
            list_of_churches.append(i)
            for col in ['song1','song2','song3','song4','song5','song6','song7','song8','song9','song10']:
                for i in df1[col]:
                    if i != 'nan':
                        for x in i.split(','):
                            song_list.append(re.sub(r"^[ ]+|[^a-zA-Z ]+", "", x).lower())
            song_df=pd.DataFrame()
            song_df['songs']=song_list
            a=song_df['songs'].value_counts().reset_index()
            #a=a[a['count']>3]
            list_dfs.append(a)
    return list_dfs, list_of_churches
#a,b= church_song_diversity(df)


In [ ]:
#this function creates a single combined dataframe of the verses
#if a verse was quoted more than once in a single serice, it only counts once
def verse_return(df):
    list1=[]
    list2=[]
    list3=[]
    list4=[]
    df_verse=pd.DataFrame()
    for i in range(0,df.shape[0]):
        verse_list_overall=[]
        verse_list_worship=[]
        verse_list_prayer=[]
        verse_list_exclusive=[]
        for col in ['1','2','3','4','5','6','7','8','9','10']:
            a=df.iloc[i][f'verse{col}']
            if len(a.split(', '))>1:
                for x in a.split(', '):
                    verse_list_overall.append(x.split(',')[0])
                    if df.iloc[i][f'song{col}']!='nan':
                        verse_list_worship.append(x.split(',')[0])
                    if df.iloc[i][f'prayer{col}']!='nan':
                        verse_list_prayer.append(x.split(',')[0])
            else:
                verse_list_overall.append(a.split(',')[0])
                if df.iloc[i][f'song{col}']!='nan':
                    verse_list_worship.append(a.split(',')[0])
                if df.iloc[i][f'prayer{col}']!='nan':
                    verse_list_prayer.append(a.split(',')[0])
        verse_list_overall=list(set(verse_list_overall))
        verse_list_worship=list(set(verse_list_worship))
        verse_list_prayer=list(set(verse_list_prayer))
        for x in verse_list_overall:
            if x in verse_list_worship:
                pass
            elif x in verse_list_prayer:
                pass
            else:
                verse_list_exclusive.append(x)

        verse_list_overall=[x for x in verse_list_overall if x!='nan']
        verse_list_worship=[x for x in verse_list_worship if x!='nan']
        verse_list_prayer=[x for x in verse_list_prayer if x!='nan']
        verse_list_exclusive=[x for x in verse_list_exclusive if x!='nan']

        list1.append(verse_list_overall)
        list2.append(verse_list_worship)
        list3.append(verse_list_prayer)
        list4.append(verse_list_exclusive)
    df_verse['overall']=list1
    df_verse['worship']=list2
    df_verse['prayer']=list3
    df_verse['exclusive']=list4
    df_verse['files']=df['files']
    df_verse['date']=df['date']
    df_verse['church_name']=df['church_name']
    return df_verse

df_verse=verse_return(df)

#what are the top verses overall?
def top_verses(df_verse):
    dfs_test1=[]
    for col in ['overall','worship','prayer','exclusive']:
        list1=[]
        for i in range(0,df_verse.shape[0]):
            a=df_verse.iloc[i]
            for x in a[col]:
                list1.append(x)
        df_test=pd.DataFrame()
        df_test[col]=list1
        df_test['files']=df_verse['files']
        df_test['date']=df_verse['date']
        df_test['church_name']=df_verse['church_name']
        dfs_test1.append(df_test[col].value_counts().reset_index())
    return dfs_test1

#what are the top books overall?
def top_books(df_verse):
    dfs_test1=[]
    for col in ['overall','worship','prayer','exclusive']:
        list1=[]
        for i in range(0,df_verse.shape[0]):
            a=df_verse.iloc[i]
            for x in a[col]:
                list1.append(str(x.rsplit(':')[0].rsplit(' ')[:-1]).replace(',','').replace("'","").replace('[','').replace(']',''))
        df_test=pd.DataFrame()
        df_test[col]=list1
        dfs_test1.append(df_test[col].value_counts().reset_index())
    return dfs_test1

#this line could be used to see the top verses from a certain timeframe
#dfs_test1=top_verses(df_verse[(df_verse['date']>'20250201')&(df_verse['date']<'20250301')])

church_book_list=[]
for i in list(df['church_name'].unique()):
    a=top_books(df_verse[df_verse['church_name']==i])[0]
    a['church']=i
    church_book_list.append(a)


In [ ]:
#this is the topic modelling done on the prayer sections using BERTopic
#example usage - https://maartengr.github.io/BERTopic/index.html
from bertopic import BERTopic

docs = []
for i in range(1,11):
    for x in df[f'prayer{i}']:
        if x!='nan':
            docs.append(x)
docs=list(set(docs))
    

topic_model = BERTopic()
topics, probs = topic_model.fit_transform(docs)

In [ ]:
#this line will explain more about the contents of a single topic
#note that topic -1 is all of the common words that don't fit into a meaningful topic, ignore topic -1
topic_model.get_topic(9)

In [ ]:
#this cell shows some of the representative documents from a topic if it isn't clear what the topic represents
a=topic_model.get_document_info(docs)
for x in a[a['Topic']==9]['Document']:
    print(x)

In [ ]:
#this cell does PCA to a dataframe and prints a visual. 
#this is made for numerical data, such as the stats on song diversity by church
#Example usage - https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN

def perform_pca_and_detect_outliers(df):
    # Standardizing the data
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(df)

    # Performing PCA
    pca = PCA(n_components=2)
    pca_result = pca.fit_transform(data_scaled)

    # Clustering using DBSCAN to detect outliers
    dbscan = DBSCAN(eps=1.5, min_samples=2)
    clusters = dbscan.fit_predict(pca_result)

    # Plotting the results
    plt.figure(figsize=(8, 6))
    plt.scatter(pca_result[:, 0], pca_result[:, 1], c=clusters, cmap='rainbow', edgecolors='k')
    plt.xlabel('Principal Component 1')
    plt.ylabel('Principal Component 2')
    plt.title('2 Component PCA of Church Song Diversity')
    plt.colorbar(label='Cluster')
    plt.show()

    # Identifying outliers (DBSCAN labels outliers as -1)
    outliers = df[clusters == -1]
    return outliers